# PyTorch Basics Tutorial
This notebook covers fundamental PyTorch concepts including tensor creation, operations, and neural network layers.

In [ ]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu
!pip install pandas numpy matplotlib scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 12.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 11.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pandas]2m5/6 [pandas]learn]


## Imports
Import the torch library for tensor operations.

In [ ]:
import torch

## Creating Tensors
A tensor is PyTorch's fundamental data structure. Here we create a 1D tensor (vector) from a Python list.

In [18]:
my_list = [1, 2, 3 , 4, 5]
my_tensor = torch.tensor(my_list, dtype=torch.float32)
my_tensor

tensor([1., 2., 3., 4., 5.])

## Tensor Properties: Shape
Check the shape of the tensor. Shape describes the dimensions and size along each dimension.

In [19]:
my_tensor.shape

torch.Size([5])

## Tensor Properties: Data Type
Check the data type of tensor elements. We used float32, which is common for neural networks.

In [20]:
my_tensor.dtype

torch.float32

## Tensor Operations: Matrix Multiplication

The `@` operator does matrix multiplication — the core operation inside every neural network layer.
Each output cell is the **dot product** of a row (left matrix) with a column (right matrix).

```
   A            B              A @ B
 ┌     ┐    ┌     ┐         ┌            ┐
 │ 1 2 │    │ 5 6 │         │ 1·5+2·8  1·6+2·9 │   ┌ 21 24 ┐
 │ 4 5 │  @ │ 8 9 │    =    │ 4·5+5·8  4·6+5·9 │ = │ 60 69 │
 └     ┘    └     ┘         └            ┘         └       ┘
```

**Shape rule:** `(m × n) @ (n × p) → (m × p)` — the inner dimensions must match.

In [24]:
my_tensor1 = torch.tensor([[1, 2, ],
                            [4, 5, ]])

my_tensor2 = torch.tensor([[5, 6, ],
                            [8, 9, ]]
                            )

print(my_tensor1 @ my_tensor2)

tensor([[21, 24],
        [60, 69]])


## Neural Network Layers

A **Linear** (fully connected) layer connects every input to every output. Below, 5 inputs → 3 output
neurons; each neuron has one weight per input plus one bias: `output = Σ(w·x) + b`.

```mermaid
graph LR
    x1 & x2 & x3 & x4 & x5 --> o1 & o2 & o3
    classDef in fill:#4c8bf5,color:#fff,stroke:#1b3b6f;
    classDef out fill:#f5a623,color:#fff,stroke:#7a4a00;
    class x1,x2,x3,x4,x5 in
    class o1,o2,o3 out
```

Params here: weights `3 × 5 = 15` + biases `3` = **18 learnable numbers**.

In [2]:
import torch.nn as nn
input_tensor = torch.tensor([[1, 2, 3, 4, 5]], dtype=torch.float32)
linear_layer = nn.Linear(in_features=5, out_features=3)
output_tensor = linear_layer(input_tensor) 
output_tensor

tensor([[4.4248, 1.3771, 4.0669]], grad_fn=<AddmmBackward0>)

A linear (fully connected) layer performs the transformation:
$$\text{output} = \text{input} \cdot \mathbf{W}^T + \mathbf{b}$$


In [30]:
linear_layer.weight

Parameter containing:
tensor([[-0.2840, -0.2452, -0.2679,  0.0188, -0.2418],
        [-0.3697, -0.1334, -0.2314,  0.0674,  0.2665],
        [ 0.2475, -0.1117, -0.3532, -0.1629, -0.0925]], requires_grad=True)

In [31]:
linear_layer.bias

Parameter containing:
tensor([ 0.3635,  0.3425, -0.0706], requires_grad=True)

#### Linear Layer: Neurons and Parameters
Understanding the structure of a neural network layer requires knowing:
- **Number of neurons (output units)**: Determines output dimension
- **Total parameters**: Number of weights and biases to learn


In [3]:
model = nn.Sequential(nn.Linear(in_features=5, out_features=3),
                      nn.Linear(in_features=3, out_features=2),
                      nn.Linear(in_features=2, out_features=2)
                      )

Layers stacked inside `nn.Sequential` (except the last) act as **hidden layers**.
Each layer's parameter count = `neurons × (inputs_per_neuron + 1 bias)`.

```mermaid
graph LR
    I["Input<br/>5 features"] --> L1["Linear<br/>3 neurons<br/>3×(5+1) = 18"]
    L1 --> L2["Linear<br/>2 neurons<br/>2×(3+1) = 8"]
    L2 --> L3["Linear<br/>2 neurons<br/>2×(2+1) = 6"]
    L3 --> O["Output<br/>2"]
```

- Layer 1: **18** · Layer 2: **8** · Layer 3: **6**
- **Total learnable parameters = 32** (verified by the code below)

In [4]:
total = 0
for parameter in model.parameters():
    total += parameter.numel()
print(total)

32


### Activation Functions

Stacking Linear layers alone is still just one big linear function. **Activation functions add
non-linearity**, letting the network learn curved decision boundaries.

<div style="display:flex;gap:40px;flex-wrap:wrap;align-items:flex-end">

<figure style="margin:0">
<figcaption><b>Sigmoid</b> — binary, squashes to (0, 1)</figcaption>
<svg width="300" height="170" viewBox="0 0 300 170" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Sigmoid curve">
  <line x1="35" y1="15" x2="35" y2="150" stroke="currentColor" stroke-width="1" opacity="0.4"/>
  <line x1="35" y1="150" x2="285" y2="150" stroke="currentColor" stroke-width="1" opacity="0.4"/>
  <line x1="35" y1="83" x2="285" y2="83" stroke="currentColor" stroke-dasharray="4 4" stroke-width="1" opacity="0.25"/>
  <path d="M35,150 C160,150 160,15 285,15" fill="none" stroke="#4c8bf5" stroke-width="2.5"/>
  <text x="18" y="20" font-size="11" fill="currentColor">1</text>
  <text x="14" y="87" font-size="11" fill="currentColor">.5</text>
  <text x="18" y="153" font-size="11" fill="currentColor">0</text>
  <text x="150" y="167" font-size="11" fill="currentColor">input →</text>
</svg>
</figure>

<figure style="margin:0">
<figcaption><b>Softmax</b> — multi-class, probs sum to 1</figcaption>
<svg width="240" height="170" viewBox="0 0 240 170" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Softmax bars">
  <line x1="30" y1="150" x2="220" y2="150" stroke="currentColor" stroke-width="1" opacity="0.4"/>
  <rect x="45"  y="55"  width="40" height="95" fill="#4c8bf5" rx="3"/>
  <rect x="105" y="112" width="40" height="38" fill="#4c8bf5" rx="3"/>
  <rect x="165" y="135" width="40" height="15" fill="#4c8bf5" rx="3"/>
  <text x="52"  y="48"  font-size="11" fill="currentColor">0.66</text>
  <text x="112" y="105" font-size="11" fill="currentColor">0.24</text>
  <text x="172" y="128" font-size="11" fill="currentColor">0.10</text>
  <text x="52"  y="164" font-size="11" fill="currentColor">cls0</text>
  <text x="112" y="164" font-size="11" fill="currentColor">cls1</text>
  <text x="172" y="164" font-size="11" fill="currentColor">cls2</text>
</svg>
</figure>

</div>

- **Sigmoid** → binary classification, output in range 0–1
- **Softmax** → multi-class classification, outputs a probability distribution (sums to 1)
- **ReLU** (`max(0, x)`) → common hidden-layer activation, cheap and effective

In [6]:
new_model = nn.Sequential(nn.Linear(in_features=5, out_features=3),
                      nn.Linear(in_features=3, out_features=2),
                      nn.Linear(in_features=2, out_features=1),
                      nn.Sigmoid()
                      )
new_model

Sequential(
  (0): Linear(in_features=5, out_features=3, bias=True)
  (1): Linear(in_features=3, out_features=2, bias=True)
  (2): Linear(in_features=2, out_features=1, bias=True)
  (3): Sigmoid()
)

In [8]:
new_model_1 = nn.Sequential(nn.Linear(in_features=5, out_features=3),
                      nn.Linear(in_features=3, out_features=2),
                      nn.Linear(in_features=2, out_features=1),
                      nn.Softmax(dim=-1) #indicate softmax is added to last dimention
                      )
new_model_1

Sequential(
  (0): Linear(in_features=5, out_features=3, bias=True)
  (1): Linear(in_features=3, out_features=2, bias=True)
  (2): Linear(in_features=2, out_features=1, bias=True)
  (3): Softmax(dim=-1)
)

### Forward Pass

Input data flows **forward** through the layers; each layer applies its weights, bias, and
activation, and the final layer produces the prediction.

```mermaid
graph LR
    X["Input<br/>x"] --> A["Linear<br/>W·x + b"] --> B["Activation<br/>σ"]
    B --> C["Linear<br/>W·x + b"] --> D["Activation<br/>σ"] --> Y["Output<br/>prediction"]
```

#### One-Hot Encoding

Converts an integer class label into a vector of 0s with a single 1 at the class index.
Networks output one score per class, so labels are represented the same way.

```
  class 0 -> [1, 0, 0]
  class 1 -> [0, 1, 0]
  class 2 -> [0, 0, 1]
```

In [9]:
import torch.nn.functional as F

F.one_hot(torch.tensor([0]),num_classes=3)

tensor([[1, 0, 0]])

In [10]:
F.one_hot(torch.tensor([1]),num_classes=3)

tensor([[0, 1, 0]])

In [11]:
F.one_hot(torch.tensor([2]),num_classes=3)

tensor([[0, 0, 1]])

### Cross-Entropy Loss

The standard loss for **classification**. It measures the gap between the predicted class scores
(logits) and the true class — high when the model is confidently wrong, near 0 when confidently right.

Below the true class is index 0, but its score (`-5.2`) is the *lowest* of the three, so the loss is
large (`≈ 9.8`). As training pushes the correct class's score up, this loss shrinks.

In [13]:
from torch.nn import CrossEntropyLoss

scores = torch.tensor([-5.2, 4.6, 0.8])
one_hot_target = torch.tensor([1, 0, 0])

loss = CrossEntropyLoss()
loss(scores.double(), one_hot_target.double())
                       

tensor(9.8222, dtype=torch.float64)

### Convex & Non-Convex Functions

The **loss landscape** is the surface we try to descend. Its shape decides how hard training is.

<div style="display:flex;gap:40px;flex-wrap:wrap;align-items:flex-end">

<figure style="margin:0">
<figcaption><b>Convex</b> — one global minimum</figcaption>
<svg width="260" height="160" viewBox="0 0 260 160" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Convex bowl">
  <path d="M20,20 Q130,210 240,20" fill="none" stroke="#4c8bf5" stroke-width="2.5"/>
  <circle cx="130" cy="118" r="5" fill="#2e9e5b"/>
  <text x="98" y="145" font-size="11" fill="currentColor">global min</text>
</svg>
</figure>

<figure style="margin:0">
<figcaption><b>Non-convex</b> — many local minima</figcaption>
<svg width="300" height="160" viewBox="0 0 300 160" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Non-convex landscape">
  <path d="M15,40 C55,150 85,60 120,110 C150,150 175,50 210,120 C235,165 265,70 290,95"
        fill="none" stroke="#4c8bf5" stroke-width="2.5"/>
  <circle cx="70"  cy="108" r="4.5" fill="#e0a53b"/>
  <circle cx="205" cy="123" r="5"   fill="#2e9e5b"/>
  <text x="40"  y="130" font-size="10" fill="currentColor">local</text>
  <text x="178" y="145" font-size="10" fill="currentColor">global</text>
</svg>
</figure>

</div>

- **Convex** → a single global minimum; gradient descent always reaches it.
- **Non-convex** (real neural networks) → many local minima; we settle for a *good enough* one.

### Backpropagation

Weights and biases start out **random**, so early predictions are wrong. Backpropagation runs the
error *backward* through the network (chain rule) to compute how much each parameter contributed to
the loss — the gradient. `loss.backward()` fills in `.grad` for every parameter.

```mermaid
graph LR
    x["Input"] --> l1["Layer 1"] --> l2["Layer 2"] --> l3["Layer 3"] --> loss["Loss"]
    loss -. "∂L/∂W₃" .-> l3
    l3 -. "∂L/∂W₂" .-> l2
    l2 -. "∂L/∂W₁" .-> l1
    linkStyle 4,5,6 stroke:#e0546b,stroke-width:2px;
```

Solid = forward pass, dashed red = gradients flowing backward. The optimizer then uses these
gradients to nudge each weight in the direction that lowers the loss.

In [14]:
test_model = nn.Sequential(
    nn.Linear(16, 8),
    nn.Linear(8, 4),
    nn.Linear(4, 2)
)

# Input: batch of 5 samples, each with 16 features
x = torch.randn(5, 16)

# Target: class labels (0 or 1)
target = torch.tensor([0, 1, 0, 1, 1])

# Forward pass
prediction = test_model(x)

# Loss function
criterion = nn.CrossEntropyLoss()

# Compute loss
loss = criterion(prediction, target)

# Backpropagation
loss.backward()

### Gradient Descent

Gradient descent uses the gradients from backprop to take a small step **downhill** on the loss
surface. The step size is the **learning rate** (`lr`); `optimizer.step()` applies one update:

<figure style="margin:0">
<figcaption><code>new_weight = old_weight − lr × gradient</code></figcaption>
<svg width="300" height="180" viewBox="0 0 300 180" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Gradient descent steps down a bowl">
  <path d="M25,25 Q150,235 275,25" fill="none" stroke="currentColor" stroke-width="2" opacity="0.5"/>
  <circle cx="55"  cy="70"  r="5" fill="#4c8bf5"/>
  <circle cx="80"  cy="105" r="5" fill="#4c8bf5"/>
  <circle cx="108" cy="130" r="5" fill="#4c8bf5"/>
  <circle cx="140" cy="142" r="5" fill="#4c8bf5"/>
  <circle cx="150" cy="145" r="6" fill="#2e9e5b"/>
  <text x="40"  y="60"  font-size="11" fill="currentColor">start</text>
  <text x="120" y="170" font-size="11" fill="currentColor">minimum ↓</text>
</svg>
</figure>

- Learning rate **too high** → overshoots and diverges.
- Learning rate **too low** → trains very slowly.
- `SGD` (Stochastic Gradient Descent) updates using one mini-batch at a time.

In [ ]:
import torch.optim as optim

optimizer = optim.SGD(model.parameters(),lr=0.001)
optimizer.step()

In [ ]:
import pandas as pd

animals = pd.read_csv('animal_dataset.csv')
animals

,animal_name,hair,feathers,eggs,milk,predator,legs,tail,type
0,sparrow,0,1,1,0,0,2,1,0
1,eagle,0,1,1,0,1,2,1,0
2,cat,1,0,0,1,1,4,1,1
3,dog,1,0,0,1,0,4,1,1
4,lizard,0,0,1,0,1,4,1,2


In [ ]:
import numpy as np

feature = animals.iloc[:, 1:-1]
x = feature.to_numpy()
x

array([[0, 1, 1, 0, 0, 2, 1],
       [0, 1, 1, 0, 1, 2, 1],
       [1, 0, 0, 1, 1, 4, 1],
       [1, 0, 0, 1, 0, 4, 1],
       [0, 0, 1, 0, 1, 4, 1]])

In [ ]:
target = animals.iloc[:, -1]
y = target.to_numpy()
y

array([0, 0, 1, 1, 2])

#### TensorDataset is used to convert data into pytorch tensors

In [ ]:
# TensorDataset pairs each input row with its label so they stay together
# when we shuffle/batch later. It turns (X, y) into an indexable dataset.
from torch.utils.data import TensorDataset

dataset = TensorDataset(
    torch.tensor(x, dtype=torch.float32),  # features -> float for the network
    torch.tensor(y, dtype=torch.long),     # class labels -> long (int) for the loss
)

# dataset[i] returns the (features, label) pair for sample i
input_samples, label_samples = dataset[0]

In [14]:
input_samples

tensor([0., 1., 1., 0., 0., 2., 1.])

In [15]:
label_samples

tensor(0)

##### DataLoader manages batching and shuffling during training

- **batch**: a small group of samples processed together in one step
- **epoch**: one full pass through the entire training dataset
- **generalization**: how well the model performs on unseen data (the real goal)

In [ ]:
# DataLoader wraps a dataset and hands it out in shuffled mini-batches.
# Each loop iteration gives one batch instead of one sample at a time.
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_inputs, batch_labels in dataloader:
    print("Input Samples:", batch_inputs)
    print("Label Samples:", batch_labels)
    print("---")

### Training Loop — Putting It All Together

Now we combine every piece into one training run on the animal dataset:
**7 input features → hidden layer → 3 classes**. Each batch runs these 5 steps in a loop:

```mermaid
graph LR
    A["zero_grad()<br/>clear old gradients"] --> B["model(x)<br/>forward pass → logits"]
    B --> C["criterion<br/>compute loss"]
    C --> D["loss.backward()<br/>backprop gradients"]
    D --> E["optimizer.step()<br/>update weights"]
    E -->|"next batch / epoch"| A
```

> **Loss choice:** `MSELoss` (mean squared error) is for *regression* (predicting a number).
> For *classification* (predicting a class) we use `CrossEntropyLoss`, which is what we do below.

In [ ]:
# Classifier for the animal dataset: 7 features -> hidden(16) -> 3 classes.
# ReLU adds non-linearity between the linear layers.
model = nn.Sequential(
    nn.Linear(in_features=7, out_features=16),
    nn.ReLU(),
    nn.Linear(in_features=16, out_features=3),
)

# CrossEntropyLoss expects raw logits (NO Softmax on the last layer) and
# integer class labels. It applies the softmax internally.
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for features, labels in dataloader:
        optimizer.zero_grad()                   # 1. reset gradients from last step
        predictions = model(features)           # 2. forward pass  -> logits
        loss = criterion(predictions, labels)   # 3. measure the error
        loss.backward()                         # 4. backprop -> compute gradients
        optimizer.step()                        # 5. update weights
        epoch_loss += loss.item()
    if epoch % 10 == 0:
        print(f"epoch {epoch:2d}  loss {epoch_loss:.4f}")